<a href="https://colab.research.google.com/github/1stzl01/Reserving/blob/main/Mack_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Measuring the Variability of Chain Ladder Reserve Estimates
### Based on the paper by Thomas Mack

This notebook illustrates Thomas Mack's "distribution-free" approach to quantifying the standard error (variability) of chain ladder reserve estimates. While the chain ladder method is popular for its simplicity, it implicitly relies on three strong statistical assumptions:
1. **Expected Value:** $E(C_{i,k+1}|C_{i1}, ..., C_{ik}) = C_{ik}f_k$ (The expected claims in the next development period depend only on the current total and a uniform development factor $f_k$).
2. **Independence:** Accident years are strictly independent.
3. **Variance:** $Var(C_{i,k+1}|C_{i1}, ..., C_{ik}) = C_{ik}\alpha_k^2$ (The variance of the next development period is proportional to the accumulated claims amount to date).

This notebook generates a synthetic run-off triangle, applies Mack's standard error formulas, and demonstrates how to build confidence intervals for the reserves. It also includes an automated diagnostic test to check for calendar-year distortions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# ==========================================
# 1. GENERATE SYNTHETIC CLAIMS DATA
# ==========================================
# Because we do not have the original data, we simulate a 10x10 cumulative run-off triangle.
# We will generate it using a base pattern and add log-normal random noise.

np.random.seed(42)
I = 10 # Number of accident years and development years

# Initialize the triangle with NaNs
C = np.full((I, I), np.nan)

# Generate initial claims for Development Year 0
C[:, 0] = np.random.uniform(1000, 5000, I)

# True underlying development factors
true_f = np.array([2.5, 1.8, 1.4, 1.2, 1.1, 1.05, 1.02, 1.01, 1.005])

# Fill the upper triangle with noise reflecting Mack's variance assumption
for i in range(I):
    for k in range(I - i - 1):
        # We simulate the next period using the true factor and some random variation
        C[i, k+1] = C[i, k] * true_f[k] * np.random.lognormal(mean=0, sigma=0.05)

# Convert to a Pandas DataFrame for nice display
triangle_df = pd.DataFrame(C,
                           index=[f"Acc_Yr_{i+1}" for i in range(I)],
                           columns=[f"Dev_Yr_{k+1}" for k in range(I)])

print("Synthetic Cumulative Claims Triangle:")
display(triangle_df.round(0))

Synthetic Cumulative Claims Triangle:


,Dev_Yr_1,Dev_Yr_2,Dev_Yr_3,Dev_Yr_4,Dev_Yr_5,Dev_Yr_6,Dev_Yr_7,Dev_Yr_8,Dev_Yr_9,Dev_Yr_10
Acc_Yr_1,2498.0,6101.0,11283.0,15434.0,18095.0,20147.0,19224.0,17988.0,17664.0,16876.0
Acc_Yr_2,4803.0,12197.0,20981.0,27370.0,35342.0,38439.0,40498.0,38468.0,37809.0,NaN
Acc_Yr_3,3928.0,9875.0,16780.0,23938.0,27875.0,30219.0,30790.0,34453.0,NaN,NaN
Acc_Yr_4,3395.0,8481.0,14479.0,21122.0,23845.0,26505.0,25233.0,NaN,NaN,NaN
Acc_Yr_5,1624.0,3799.0,6906.0,10033.0,12143.0,13280.0,NaN,NaN,NaN,NaN
Acc_Yr_6,1624.0,3999.0,6686.0,9029.0,10588.0,NaN,NaN,NaN,NaN,NaN
Acc_Yr_7,1232.0,3248.0,5948.0,7624.0,NaN,NaN,NaN,NaN,NaN,NaN
Acc_Yr_8,4465.0,11344.0,20030.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Acc_Yr_9,3404.0,8228.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Acc_Yr_10,3832.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 1: Calculate Standard Age-to-Age Factors ($f_k$)
The traditional chain ladder method estimates the development factor $f_k$ as the volume-weighted average of the observed individual development factors.
Formula: $\hat{f}_k = \frac{\sum_{j=1}^{I-k} C_{j,k+1}}{\sum_{j=1}^{I-k} C_{j,k}}$

In [ ]:
# ==========================================
# 2. CALCULATE DEVELOPMENT FACTORS & PROJECT ULTIMATE CLAIMS
# ==========================================

f_k = np.zeros(I - 1)

# Calculate f_k for each development column
for k in range(I - 1):
    # Sum all available claims in period k+1 and divide by sum of claims in period k
    # We only sum up to row I-k-1 (the last row with data in column k+1)
    sum_num = np.nansum(C[:I-k-1, k+1])
    sum_den = np.nansum(C[:I-k-1, k])
    f_k[k] = sum_num / sum_den

print("Estimated Development Factors (f_k):")
print(np.round(f_k, 3))

# Complete the lower half of the triangle to project ultimate claims
C_full = np.copy(C)
for i in range(1, I):
    for k in range(I - i, I):
        C_full[i, k] = C_full[i, k-1] * f_k[k-1]

# Calculate Individual Reserves (Ultimate - Latest Known)
latest_known = np.array([C[i, I-i-1] for i in range(I)])
ultimate_claims = C_full[:, I-1]
reserves = ultimate_claims - latest_known
total_reserve = np.sum(reserves)

print("\nProjected Outstanding Reserves by Accident Year:")
print(np.round(reserves, 0))
print(f"\nOverall Estimated Reserve: {total_reserve:,.0f}")

Estimated Development Factors (f_k):
[2.494 1.746 1.379 1.196 1.096 1.004 1.004 0.983 0.955]

Projected Outstanding Reserves by Accident Year:
[    0. -1687. -2110. -1442.  -712.   397.  1837. 14248. 16357. 24727.]

Overall Estimated Reserve: 51,615


### Step 2: Calculate Variance Parameters ($\alpha_k^2$)
Mack proves that the volume-weighted chain ladder implies the variance of the next period is proportional to the current accumulated claims.
We estimate the proportionality constant $\alpha_k^2$ using the unbiased estimator:
$\alpha_k^2 = \frac{1}{I-k-1} \sum_{i=1}^{I-k} C_{i,k} \left( \frac{C_{i,k+1}}{C_{i,k}} - f_k \right)^2$

Because the formula cannot estimate the parameter for the final development period ($\alpha_{I-1}^2$) as there is only one observation, Mack suggests extrapolating it. We will extrapolate it using $\alpha_{I-1}^2 = \min(\frac{\alpha_{I-2}^4}{\alpha_{I-3}^2}, \alpha_{I-2}^2)$.


In [ ]:
# ==========================================
# 3. CALCULATE VARIANCE PARAMETERS (alpha_k^2)
# ==========================================

alpha_2 = np.zeros(I - 1)

# Calculate alpha_2 for k = 0 to I-3
for k in range(I - 2):
    sum_var = 0
    # Loop over all rows that have data for BOTH k and k+1
    for i in range(I - k - 1):
        individual_factor = C[i, k+1] / C[i, k]
        sum_var += C[i, k] * (individual_factor - f_k[k])**2

    # Divide by the degrees of freedom (I - k - 1 in Python's 0-indexed logic)
    alpha_2[k] = sum_var / (I - k - 1 - 1) # i.e. I - k - 2

# Extrapolate the final variance parameter (alpha_{I-1}^2)
# Using Mack's suggested heuristic to prevent instability
alpha_2[I-2] = min((alpha_2[I-3]**2) / alpha_2[I-4], alpha_2[I-3])

print("Estimated Variance Parameters (alpha_k^2):")
print(np.round(alpha_2, 2))

Estimated Variance Parameters (alpha_k^2):
[1.4200e+01 2.5340e+01 5.7800e+01 7.6520e+01 4.4300e+00 7.4230e+01
 3.0767e+02 1.0000e-02 0.0000e+00]


### Step 3: Calculate Standard Errors
Mack establishes closed-form solutions for the standard errors without assuming a specific underlying distribution.
The squared standard error for an individual accident year is:
$(s.e.(R_i))^2 = \hat{C}_{i,I}^2 \sum_{k=I+1-i}^{I-1} \frac{\alpha_k^2}{f_k^2} \left( \frac{1}{\hat{C}_{i,k}} + \frac{1}{\sum_{j=1}^{I-k} C_{j,k}} \right)$

The overall reserve standard error accounts for the correlations between different accident years generated by using the same development factors.

In [ ]:
# ==========================================
# 4. CALCULATE STANDARD ERRORS
# ==========================================

se_R_i = np.zeros(I)
se_R_i_squared = np.zeros(I)

# Calculate individual standard errors for each accident year
for i in range(1, I):
    sum_terms = 0
    # Sum over the future development periods for this accident year
    for k in range(I - i, I - 1):
        term1 = alpha_2[k] / (f_k[k]**2)
        # 1 / C_hat_{i,k} + 1 / Sum(C_{j,k} for known j)
        term2 = (1 / C_full[i, k]) + (1 / np.nansum(C[:I-k-1, k]))
        sum_terms += term1 * term2

    se_R_i_squared[i] = (C_full[i, I-1]**2) * sum_terms
    se_R_i[i] = np.sqrt(se_R_i_squared[i])

# Calculate Overall Standard Error (including covariances between accident years)
covariance_term = 0
for i in range(1, I):
    for j in range(i + 1, I):
        sum_cov = 0
        for k in range(I - i, I - 1):
            sum_cov += (2 * alpha_2[k]) / (f_k[k]**2 * np.nansum(C[:I-k-1, k]))
        covariance_term += C_full[i, I-1] * C_full[j, I-1] * sum_cov

se_overall = np.sqrt(np.sum(se_R_i_squared) + covariance_term)

# Print Summary
results = pd.DataFrame({
    'Reserve': reserves,
    'Std Error': se_R_i,
    'CV (%)': (se_R_i / np.where(reserves==0, 1, reserves)) * 100
})
print("Reserve Estimates and Variability by Accident Year:")
display(results.round(1).iloc[1:]) # Drop Acc Yr 1 (fully developed)

print(f"\nOverall Reserve: {total_reserve:,.0f}")
print(f"Overall Standard Error: {se_overall:,.0f} (CV: {(se_overall/total_reserve)*100:.1f}%)")


Reserve Estimates and Variability by Accident Year:


,Reserve,Std Error,CV (%)
1,-1687.3,0.0,-0.0
2,-2110.3,0.2,-0.0
3,-1441.6,17.8,-1.2
4,-711.6,2036.3,-286.2
5,397.2,2099.7,528.7
6,1836.8,1943.8,105.8
7,14247.6,4470.9,31.4
8,16357.2,3841.1,23.5
9,24726.7,4303.5,17.4



Overall Reserve: 51,615
Overall Standard Error: 10,648 (CV: 20.6%)


### Step 4: Testing Assumption 2 (Calendar Year Effects)
A core chain ladder assumption is that accident years are independent. This is violated if there are calendar year effects (e.g., a change in claims handling speed or a surge in inflation affecting a specific calendar year diagonal).

Mack proposes a "Distribution-Free" test for this:
1. For each column $k$, calculate the median development factor.
2. Label each observed factor as "Larger" (L) or "Smaller" (S) than the median.
3. For each calendar year diagonal, count the number of L's and S's.
4. If a diagonal has a significant preponderance of L's or S's (measured via a binomial distribution probability), it suggests a calendar year distortion.

In [ ]:
# ==========================================
# 5. DIAGNOSTIC TEST: "LARGE/SMALL" CALENDAR YEAR EFFECT
# ==========================================

# Create an array to hold the "L", "S", or "*" (median/missing) labels
LS_matrix = np.full((I-1, I-1), "*", dtype=object)

# Assign L and S based on column medians
for k in range(I - 1):
    # Extract known factors for column k
    factors = []
    for i in range(I - k - 1):
        factors.append(C[i, k+1] / C[i, k])

    if len(factors) > 0:
        median_f = np.median(factors)
        for i in range(I - k - 1):
            val = C[i, k+1] / C[i, k]
            if val > median_f:
                LS_matrix[i, k] = "L"
            elif val < median_f:
                LS_matrix[i, k] = "S"
            else:
                LS_matrix[i, k] = "*" # Median itself is discarded in odd-numbered samples

print("L/S Matrix by Accident Year (Rows) and Development Year (Cols):")
display(pd.DataFrame(LS_matrix))

# Count L's and S's along Calendar Year Diagonals (j = i + k)
# We test j from 1 to I-2 (leaving out the very tips)
calendar_years = {}
for j in range(1, I - 1):
    L_count = 0
    S_count = 0

    for i in range(I - 1):
        for k in range(I - 1):
            if i + k == j:
                if LS_matrix[i, k] == "L":
                    L_count += 1
                elif LS_matrix[i, k] == "S":
                    S_count += 1

    n_total = L_count + S_count
    z_j = min(L_count, S_count)

    # Calculate Binomial Cumulative Probability: P(Z <= z_j) given p=0.5
    if n_total > 0:
        prob = stats.binom.cdf(z_j, n_total, 0.5) * 2 # Two-tailed test roughly
        calendar_years[f"Cal_Yr_{j}"] = {"L": L_count, "S": S_count, "n": n_total, "Z": z_j, "Prob(Z<=z)": prob}

print("\nCalendar Year Diagnostics:")
cal_df = pd.DataFrame(calendar_years).T
display(cal_df)

print("\nInterpretation:")
print("If 'Prob(Z<=z)' is very small (e.g., < 0.10), it indicates that either 'L' or 'S' factors dominate this calendar year.")
print("If such a trend exists, Mack suggests reducing the weight of these specific factors in the chain ladder calculation.")

L/S Matrix by Accident Year (Rows) and Development Year (Cols):


,0,1,2,3,4,5,6,7,8
0,S,L,*,S,L,S,S,S,*
1,L,S,S,L,S,L,*,L,*
2,L,S,L,S,S,L,L,*,*
3,*,S,L,S,L,S,*,*,*
4,S,L,L,L,*,*,*,*,*
5,S,S,S,L,*,*,*,*,*
6,L,L,S,*,*,*,*,*,*
7,L,L,*,*,*,*,*,*,*
8,S,*,*,*,*,*,*,*,*



Calendar Year Diagnostics:


,L,S,n,Z,Prob(Z<=z)
Cal_Yr_1,2.0,0.0,2.0,0.0,0.500000
Cal_Yr_2,1.0,1.0,2.0,1.0,1.500000
Cal_Yr_3,0.0,3.0,3.0,0.0,0.250000
Cal_Yr_4,3.0,2.0,5.0,2.0,1.000000
Cal_Yr_5,2.0,4.0,6.0,2.0,0.687500
Cal_Yr_6,3.0,4.0,7.0,3.0,1.000000
Cal_Yr_7,5.0,2.0,7.0,2.0,0.453125
Cal_Yr_8,4.0,3.0,7.0,3.0,1.000000



Interpretation:
If 'Prob(Z<=z)' is very small (e.g., < 0.10), it indicates that either 'L' or 'S' factors dominate this calendar year.
If such a trend exists, Mack suggests reducing the weight of these specific factors in the chain ladder calculation.
